# Gold Layer - Grid Incident Summary

## Purpose
Aggregate event-level incidents to **daily summary by region and severity** for operational tracking.

## Business Question
**"What operational incidents happened, where, and how severe were they?"**

## Output Table
`vattenfall_dev.gold.gold_grid_incident_summary`

**Grain:** `event_day` × `region` × `severity_band`

**Measures:**
* Incident count
* Total outage duration (minutes)
* Average outage duration (minutes)
* Elevated incident count
* Critical incident count
* Total customers affected
* Average customers per incident

## Source
`vattenfall_dev.refined.silver_grid_events` (165 event-level records)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("✅ Setup complete")

In [0]:
# Load silver grid events data
df_silver = spark.table("vattenfall_dev.refined.silver_grid_events")

print(f"✅ Loaded {df_silver.count():,} grid event records")
print(f"   Date range: {df_silver.agg(F.min('event_day')).first()[0]} to {df_silver.agg(F.max('event_day')).first()[0]}")
print(f"   Regions: {df_silver.select('region_normalized').distinct().count()}")
print(f"   Severity levels: {df_silver.select('severity').distinct().count()}")

# Show sample
display(df_silver.select(
    'event_day', 'region_normalized', 'severity', 'event_type',
    'duration_minutes', 'affected_customers', 'severity_band'
).limit(5))

In [0]:
# Aggregate to daily level by region and severity
df_daily = df_silver.groupBy(
    "event_day",
    F.col("region_normalized").alias("region"),
    F.col("severity_band").alias("severity_band")
).agg(
    # Incident counts
    F.count("*").alias("incident_count"),
    F.sum(F.when(F.col("severity_band").isin(["HIGH", "CRITICAL"]), 1).otherwise(0)).alias("elevated_incident_count"),
    F.sum(F.when(F.col("severity_band") == "CRITICAL", 1).otherwise(0)).alias("critical_incident_count"),
    
    # Duration metrics
    F.round(F.sum("duration_minutes"), 2).alias("total_duration_minutes"),
    F.round(F.avg("duration_minutes"), 2).alias("avg_duration_minutes"),
    F.round(F.max("duration_minutes"), 2).alias("max_duration_minutes"),
    
    # Customer impact
    F.sum("affected_customers").alias("total_customers_affected"),
    F.round(F.avg("affected_customers"), 0).alias("avg_customers_per_incident"),
    F.max("affected_customers").alias("max_customers_affected"),
    
    # Event type distribution
    F.count_distinct("event_type").alias("unique_event_types")
).orderBy("event_day", "region", "severity_band")

print(f"✅ Aggregated to {df_daily.count()} daily incident summaries")
print(f"   Grain: event_day × region × severity_band")

# Show sample
display(df_daily.limit(10))

In [0]:
# Ensure gold schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS vattenfall_dev.gold")

# Write to Gold layer
table_name = "vattenfall_dev.gold.gold_grid_incident_summary"

df_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Successfully wrote to {table_name}")
print(f"   Records: {spark.table(table_name).count():,}")
print(f"   Columns: {len(spark.table(table_name).columns)}")

In [0]:
# Reload for validation
df_validate = spark.table("vattenfall_dev.gold.gold_grid_incident_summary")

print("=" * 70)
print("GOLD TABLE VALIDATION")
print("=" * 70)

# 1. Check grain uniqueness
print("\n1. GRAIN VALIDATION (event_day × region × severity_band)")
print("-" * 70)
total_rows = df_validate.count()
unique_combos = df_validate.select("event_day", "region", "severity_band").distinct().count()
print(f"   Total rows: {total_rows}")
print(f"   Unique day-region-severity combinations: {unique_combos}")
print(f"   ✅ Grain is unique: {total_rows == unique_combos}")

# 2. Check for nulls in key columns
print("\n2. NULL CHECK")
print("-" * 70)
for col in ["event_day", "region", "severity_band", "incident_count"]:
    null_count = df_validate.filter(F.col(col).isNull()).count()
    status = "✅ PASS" if null_count == 0 else f"❌ {null_count} nulls"
    print(f"   {col:.<30} {status}")

# 3. Date coverage
print("\n3. DATE COVERAGE")
print("-" * 70)
date_stats = df_validate.agg(
    F.min("event_day").alias("start_date"),
    F.max("event_day").alias("end_date"),
    F.countDistinct("event_day").alias("unique_days")
).first()
print(f"   Start date: {date_stats['start_date']}")
print(f"   End date: {date_stats['end_date']}")
print(f"   Unique days: {date_stats['unique_days']}")

# 4. Regional incident summary
print("\n4. REGIONAL INCIDENT SUMMARY")
print("-" * 70)
df_validate.groupBy("region").agg(
    F.sum("incident_count").alias("total_incidents"),
    F.sum("critical_incident_count").alias("critical_incidents"),
    F.sum("total_customers_affected").alias("customers_affected"),
    F.round(F.avg("avg_duration_minutes"), 1).alias("avg_duration")
).orderBy(F.col("total_incidents").desc()).show()

# 5. Severity distribution
print("\n5. SEVERITY DISTRIBUTION")
print("-" * 70)
df_validate.groupBy("severity_band").agg(
    F.sum("incident_count").alias("total_incidents"),
    F.round(F.avg("avg_duration_minutes"), 1).alias("avg_duration"),
    F.sum("total_customers_affected").alias("customers_affected")
).orderBy(F.col("total_incidents").desc()).show()

# 6. Top 10 worst incident days (by customer impact)
print("\n6. TOP 10 WORST INCIDENT DAYS (by customer impact)")
print("-" * 70)
df_validate.orderBy(F.col("total_customers_affected").desc()).limit(10).select(
    "event_day", "region", "severity_band", "incident_count",
    "critical_incident_count", "total_customers_affected", "total_duration_minutes"
).show(truncate=False)

print("\n" + "=" * 70)
print("✅ Gold table validation complete")
print("=" * 70)

---

## Business Question: "What operational incidents happened, where, and how severe were they?"

### Executive Summary (January 1-15, 2024)

Analysis of **165 grid incidents** aggregated to **97 daily summaries** across 4 Nordic regions reveals significant regional variability and severity-driven customer impact:

### Regional Incident Distribution

**🔴 Sweden - Highest Incident Volume**
* **60 incidents** (36% of all events)
* **169,145 customers affected** (39% of total impact)
* **Average duration: 148.1 minutes** (longest average)
* **Pattern:** Most frequent incidents, longest restoration times
* Sweden's extensive grid infrastructure shows elevated operational stress

**🟡 Finland - Second-Highest Impact**
* **47 incidents** (28% of all events)
* **115,327 customers affected** (27% of total impact)
* **Average duration: 116.3 minutes**
* Balanced frequency-duration profile

**🟢 Norway - Moderate Activity**
* **30 incidents** (18% of all events)
* **88,381 customers affected** (21% of total impact)
* **Average duration: 90.6 minutes** (shortest average)
* Lower incident count but **worst single-day event** (Jan 4: 22,067 customers)

**🟢 Denmark - Lowest Impact**
* **28 incidents** (17% of all events)
* **57,809 customers affected** (13% of total impact)
* **Average duration: 111.9 minutes**
* Most stable grid operations

### Severity Analysis

**Critical Priority Incidents Dominate:**
* **98 incidents** (59% of all events)
* **360,247 customers affected** (84% of total impact)
* **Average duration: 126.5 minutes**
* Critical events drive the overwhelming majority of customer impact

**High Priority - Longest Restoration Times:**
* **14 incidents** (only 8% of events)
* **Average duration: 227.5 minutes** (nearly 2× critical priority)
* **16,849 customers affected**
* Suggests complex technical issues requiring extended restoration efforts

**Medium Priority - Routine Operations:**
* **49 incidents** (30% of events)
* **Average duration: 72.6 minutes** (shortest)
* **53,323 customers affected**
* Primarily planned maintenance with controlled impact

**Low Priority - Minimal Impact:**
* **4 incidents** only
* **243 customers affected** (negligible)
* Minor operational events

### Top 10 Worst Incident Days (All Critical Priority)

| Rank | Date | Region | Incidents | Customers | Duration (min) |
|------|------|--------|-----------|-----------|----------------|
| 1 | Jan 4 | Norway | 6 | 22,067 | 982 |
| 2 | Jan 10 | Sweden | 5 | 20,103 | 441 |
| 3 | Jan 5 | Sweden | 5 | 17,097 | 656 |
| 4 | Jan 11 | Finland | 4 | 15,429 | 574 |
| 5 | Jan 7 | Sweden | 4 | 15,008 | 410 |

**Critical Insight:** Sweden appears in 7 of top 10 worst days - consistent operational stress pattern requiring investigation.

### Business Implications

✅ **Sweden Requires Immediate Attention:** 36% of all incidents, longest durations, dominates worst-day rankings - prioritize grid resilience investments and preventive maintenance.

✅ **Critical Priority Incidents Drive 84% of Customer Impact:** Focus incident prevention strategies on high-severity scenarios rather than distributing resources evenly.

✅ **High Priority Incidents Have 2× Restoration Time:** These 14 incidents suggest complex technical challenges - warrant root cause analysis and specialized response protocols.

✅ **Norway's Jan 4 Event (22,067 customers):** Single worst day affecting 25% of Norway's total 2-week impact - investigate contributing factors (weather, equipment failure, maintenance timing).

✅ **Denmark Shows Best Operational Stability:** Lowest incident count and customer impact - benchmark best practices for other regions.

